# 08 - Step 16 (Novelty 2): Vision-Language Models (`OpenAI CLIP ViT-B/16`) on LIGO Glitches
**L4 Group 23: Ismail Nooruddin, Munir Tariq, Ramsha Arshad**  
*CS/CE 351/363 - Artificial Intelligence, Fall 2026, Habib University*  
*Project Title: Classifying LIGO Glitches: Comparing CNNs and Simple Features*

---

### How to Run on Google Colab (3 Clicks — ~6–10 Minutes Total!)
1. Go to **`Runtime → Change runtime type`** and select **`L4 GPU`** *(fastest)* or **`T4 GPU`** *(works even on Free Colab!)*.
2. Click **`Runtime → Run all`** (top menu).
3. Wait ~6–10 minutes. At the end, **`step16_results.zip`** (~15 MB) will automatically download — drop it into your local `AI Project/` folder, and also save this executed notebook (`File → Download → Download .ipynb`) to replace `notebooks/08_vlm_clip_colab.ipynb`!

### Scientific Objectives of Step 16 (Apples-to-Apples vs. Step 15 ImageNet `ViT-B/16`)
* **Part A (True Zero-Shot Classification, `0` LIGO training images):** Match the `951` unseen test spectrograms directly against **Basic Class-Name Prompts** vs. **Domain-Engineered LIGO Physics Prompt Ensembles** (testing both Multi-View RGB and web-familiar `1.0s` Viridis colormapping).
* **Part B (Frozen CLIP Visual Representations + Lightweight Probes):** Extract frozen `512`-dim CLIP embeddings for all `6,340` glitches once (saved to `clip_embeddings_frozen.npz` for Step 17 Multimodal Fusion) and train fast **Linear** and **2-Layer MLP** probes with `0` backbone fine-tuning.
* **Part C1 (Prompt-Initialized End-to-End Fine-Tuned `CLIP ViT-B/16`):** Initialize a cosine classifier directly from our physics text prototypes (WiSE-FT / Prompt-Init paradigm) and gently fine-tune on the exact Seed `42` split, saving fine-tuned embeddings (`clip_embeddings_finetuned.npz`).
* **Part C2 (5-Seed Cross-Detector Transfer & Few-Shot Adaptation, `H1 ↔ L1`):** Compare frozen CLIP embeddings (`0%` Text Zero-Shot, `0%` Source Probe, `5%/10%/20%` Target-Adapted Probes, `100%` Native) against Step 13 RF/CNN and Step 15 ResNet-18 across seeds `42–46`.

In [ ]:
# Step 16.1: Verify Colab GPU, Sync Repository, and Check Hugging Face `transformers`
import os
import subprocess
import sys
import torch

print(f"PyTorch: {torch.__version__} | CUDA Available: {torch.cuda.is_available()}")
assert torch.cuda.is_available(), "No GPU detected! Go to Runtime -> Change runtime type -> Select L4 or T4 GPU."
gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
print(f"Active GPU: {gpu_name} ({vram_gb:.1f} GB VRAM)")

REPO_URL = "https://github.com/IsmailNooruddin/ligo-glitch-classification.git"
REPO_DIR = "/content/ligo-glitch-classification"

if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=True)

os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

try:
    import transformers
    print(f"Hugging Face transformers already installed: v{transformers.__version__}")
except ImportError:
    print("Installing Hugging Face transformers...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "transformers"], check=True)
    import transformers
    print(f"Installed transformers: v{transformers.__version__}")

from src.models.vlm_models import CLIP_MODEL_ID, CLIPVisionTextWrapper, PHYSICS_PROMPT_ENSEMBLE
print(f"Working Dir: {os.getcwd()}")
print(f"Verified Step 16 modules! Target VLM: {CLIP_MODEL_ID} ({len(PHYSICS_PROMPT_ENSEMBLE)} physics prompt classes)")

### Explanation of Cell 1: GPU Hardware Verification, Repo Sync & VLM Dependencies
- **What this code does:** Verifies CUDA GPU attachment (`L4` or `T4`), clones/pulls our public repository `ligo-glitch-classification` into `/content/`, confirms Hugging Face `transformers` is available, and validates clean import of `src/models/vlm_models.py`.
- **Why this matters:** Keeps Google Colab locked to the exact same leak-free splits (`split_indices.json`), metrics (`metrics.py`), and modular code tested locally.

In [ ]:
# Step 16.2: Fast Fetch & Integrity Check for trainingsetv1d1.h5 (~2.84 GiB)
import os
import subprocess
import h5py

H5_PATH = "/content/ligo-glitch-classification/trainingsetv1d1.h5"
ZENODO_URL = "https://zenodo.org/records/1486046/files/trainingsetv1d1.h5?download=1"
MIN_BYTES = int(2.5 * 1024 ** 3)  # Expect ~2.84 GiB

if not os.path.exists(H5_PATH) or os.path.getsize(H5_PATH) < MIN_BYTES:
    print("Downloading Gravity Spy HDF5 dataset (~2.84 GiB) directly into Colab...")
    ret = subprocess.run(["wget", "-q", "--show-progress", "-O", H5_PATH, ZENODO_URL])
    if ret.returncode != 0 or not os.path.exists(H5_PATH) or os.path.getsize(H5_PATH) < MIN_BYTES:
        print("wget fallback -> using curl...")
        subprocess.run(["curl", "-L", "-o", H5_PATH, ZENODO_URL], check=True)
else:
    print(f"Dataset already present: {H5_PATH}")

size_gib = os.path.getsize(H5_PATH) / (1024 ** 3)
with h5py.File(H5_PATH, "r") as f:
    n_groups = len(list(f.keys()))
print(f"Verified HDF5 integrity: {size_gib:.2f} GiB | {n_groups} root class groups found.")
assert n_groups == 22, f"Expected 22 HDF5 root class groups, got {n_groups}!" 

### Explanation of Cell 2: Automated High-Speed Dataset Fetch & Verification
- **What this code does:** Fetches `trainingsetv1d1.h5` (~`2.84 GiB`) directly from Zenodo record `1486046` over Google Colab's high-speed network (if not already cached) and confirms all `22` root HDF5 class groups open cleanly.
- **Why this matters:** Ensures zero corrupt downloads and eliminates manual multi-gigabyte uploads from local Wi-Fi.

In [ ]:
# Step 16.3: Run Complete Step 16 VLM Pipeline (Parts A, B, C1 & C2 — ~6 to 10 mins on GPU)
import os

os.chdir("/content/ligo-glitch-classification")
!python -u src/experiments/run_vlm_clip.py

assert os.path.exists("step16_results.zip"), "Expected step16_results.zip was not created!"
print(f"\nVerified output archive ready: step16_results.zip ({os.path.getsize('step16_results.zip') / (1024**2):.2f} MB)")

### Explanation of Cell 3: Full GPU Execution of Step 16 (`CLIP ViT-B/16`)
- **What this code does:** Runs `src/experiments/run_vlm_clip.py` end-to-end on the GPU with CUDA mixed precision (`AMP`):
  1. **Part A (Zero-Shot):** Evaluates `CLIP ViT-B/16` with `0` LIGO training images across Basic Prompts (RGB), Physics Prompt Ensemble (RGB), and Physics Prompt Ensemble (`1.0s` Viridis colormap).
  2. **Part B (Frozen Embeddings + Probes):** Extracts `512`-dim frozen CLIP embeddings for all `6,340` glitches once (`clip_embeddings_frozen.npz`) and fits tuned Linear & 2-Layer MLP probes.
  3. **Part C1 (Prompt-Initialized Fine-Tuning):** Initializes the cosine head from physics text prototypes and gently fine-tunes `CLIP ViT-B/16` on the Seed `42` pooled split, saving fine-tuned embeddings (`clip_embeddings_finetuned.npz`).
  4. **Part C2 (5-Seed `H1 ↔ L1` Cross-Detector Few-Shot):** Evaluates frozen CLIP embeddings across `0%` Text Zero-Shot, `0%` Source Probe, `5%/10%/20%` Target-Adapted Probes, and `100%` Native Target over seeds `42–46`.

In [ ]:
# Step 16.4: Inspect Pooled VLM Results (Parts A, B & C1) & Visuals
import pandas as pd
from IPython.display import display, Image

pooled_df = pd.read_csv("outputs/tables/vlm_clip_pooled_results.csv")
zs_df = pd.read_csv("outputs/tables/vlm_clip_zeroshot_details.csv")
per_cls_df = pd.read_csv("outputs/tables/vlm_clip_per_class_f1.csv")

print("=" * 92)
print(" STEP 16 POOLED LEADERBOARD: OPENAI CLIP ViT-B/16 PARADIGMS (UNSEEN TEST SET, N=951)")
print("=" * 92)
display(pooled_df)

print("\n=== PART A: TRUE ZERO-SHOT TOP-1 vs TOP-3 RECALL PER CLASS (0 LIGO LABELS) ===")
display(zs_df)

print("\n=== PER-CLASS F1 PROGRESSION ACROSS CLIP PARADIGMS ===")
display(per_cls_df)

print("\n--- Unified Benchmark: Prior Models vs. CLIP ViT-B/16 ---")
display(Image(filename="outputs/figures/vlm_clip_benchmark.png"))

print("\n--- Zero-Shot Per-Class Recall & Best Fine-Tuned CLIP Confusion Matrix ---")
display(Image(filename="outputs/figures/vlm_clip_zeroshot_and_cm.png"))

### Explanation of Cell 4: Pooled VLM Analysis — Zero-Shot, Frozen Probes & Text-Init Fine-Tuning
- **What to inspect above:**
  - Compare **Basic Prompts** vs. **Physics Prompt Ensemble** and **Multi-View RGB** vs. **1.0s Viridis** in Part A to see how domain prompt engineering and web-familiar colormaps affect zero-shot recognition (`0` training images).
  - Compare **Frozen Linear/MLP Probes** (zero backbone fine-tuning) against our Step 9 Random Forest (`0.9376`) and Step 12 4-Layer CNN (`0.9466`).
  - Compare **Prompt-Initialized Fine-Tuned `CLIP ViT-B/16`** directly against Step 15's **ImageNet-Pretrained `ViT-B/16` (`0.9608`)** for an exact apples-to-apples test of Vision-Language vs. ImageNet pretraining!

In [ ]:
# Step 16.5: Inspect Part C2 Cross-Detector Few-Shot Results (5 Seeds, H1 <-> L1)
import pandas as pd
from IPython.display import display, Image

cd_summary_df = pd.read_csv("outputs/tables/vlm_clip_cross_detector_summary.csv")
cd_per_cls_df = pd.read_csv("outputs/tables/vlm_clip_cross_detector_per_class.csv")

print("=" * 92)
print(" PART C2: 5-SEED CROSS-DETECTOR TRANSFER & FEW-SHOT ADAPTATION (FROZEN CLIP EMBEDDINGS)")
print("=" * 92)
display(cd_summary_df)

print("\n=== PER-CLASS CROSS-DETECTOR RECOVERY ON FROZEN CLIP EMBEDDINGS ===")
display(cd_per_cls_df)

print("\n--- Cross-Detector Comparison: Frozen CLIP Probe vs. Step 13 & Step 15 ---")
display(Image(filename="outputs/figures/vlm_clip_cross_detector_curve.png"))

### Explanation of Cell 5: Cross-Detector Generalization of Frozen CLIP Representations
- **What to inspect above:**
  - Examine how well a **completely frozen** Vision-Language representation generalizes across physical LIGO observatories (`H1 → L1` and `L1 → H1`) using only a fast linear probe—with zero GPU fine-tuning of the backbone!
  - Compare the `0% (Source Probe)` and `5% / 10% / 20% Adapted Probe` curves directly against Step 13 Zero-Shot RF (`0.6355` / `0.7101`), Step 13 4-Layer CNN (`0.7357` / `0.7095`), and Step 15 Fine-Tuned `ResNet-18` (`0.8512 → 0.9426`).

In [ ]:
# Step 16.6: Verify Archive Contents & Download `step16_results.zip` (~15 MB)
import os
import zipfile

ZIP_PATH = "/content/ligo-glitch-classification/step16_results.zip"
assert os.path.exists(ZIP_PATH), f"Missing {ZIP_PATH}!"

with zipfile.ZipFile(ZIP_PATH, "r") as zf:
    names = zf.namelist()
    print(f"Archive verified: {len(names)} files ({os.path.getsize(ZIP_PATH)/(1024**2):.2f} MB):")
    for n in names:
        print(f"  - {n}")

try:
    from google.colab import files
    files.download(ZIP_PATH)
    print("\nBrowser download triggered for step16_results.zip!")
except ImportError:
    print(f"Not in Colab UI; archive saved at {ZIP_PATH}")

### Explanation of Cell 6: Bringing Step 16 Results & Embeddings Home
- **What this code does:** Verifies all `13` generated artifacts inside `step16_results.zip` (8 CSV tables, 3 high-resolution PNG charts, and both compact `float16` CLIP embedding archives `clip_embeddings_frozen.npz` & `clip_embeddings_finetuned.npz` for Step 17 Multimodal Fusion) and automatically downloads the zip to your laptop.
- **Next Action:** Drop `step16_results.zip` into your local `AI Project/` folder, download this executed notebook (`File → Download → Download .ipynb`) to replace `notebooks/08_vlm_clip_colab.ipynb`, and tell your assistant to extract, analyze, and integrate Step 16!